In [1]:
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Conv2D, MaxPooling2D, Flatten, SimpleRNN, Reshape
from tensorflow.keras.callbacks import EarlyStopping, Callback
from tensorflow.keras.optimizers import Adam
import time
import warnings; warnings.filterwarnings('ignore')

In [2]:
class EpochTimer(Callback):
    def on_train_begin(self, logs = None): self.times = []
    def on_epoch_begin(self, epoch, logs = None): self._start = time.time()
    def on_epoch_end(self, epoch, logs = None): self.times.append(time.time() - self._start)

# Task 1
### Data Preparation and Preprocessing
- Load the fashion-MNIST dataset (training and testing) using the `load_data()` method from the `fmnist` package
- Convert the cloth items class labels into one-hot encoded format with 10 output classes
- Rescale inputs to the range $[0, 1]$
- Display the shape of the training and testing datasets

In [3]:
# Task 1: Data Preparation and Preprocessing
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Rescale to [0, 1]
X_train = X_train / 255.0
X_test  = X_test  / 255.0

# One-hot encode labels
y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
X_train: (60000, 28, 28) | y_train: (60000, 10)
X_test:  (10000, 28, 28)  | y_test:  (10000, 10)


# Task 2
### FCFNN
- Reshape each image from 28 × 28 into a 1D vector of length 784
- Build an FCFNN using the `Sequential()` API from `keras` using a suitable combination of the following layers
  - `Dense()` layers with suitable number of  neurons
  - `Dropout()` layers with a suitable rates
  - `Dense()` output layer with 10 neurons and `'softmax'` activation
  - Begin with ReLU activation for all layers except the output layer
- Study the model architecture using the `summary()` method
- Compile the model using `Adam()` optimizer, `'categorical_crossentropy'` loss, `'accuracy'` as evaluation metric
- Train the model for suitable number of epochs with a suitable batch size, and provide the validation data separately during training
- Limit training of the model using `EarlyStopping()` from `keras` with a suitable tolerance
- Feel free to change the architecture of the model (number of layers, number of neurons, activation functions, batch size, number of epochs, early stopping specifics, optimizer learning rate, and so on) and try to improve the model
- Note down the final specifics of the model such as parameter count, average training time per epoch, and performance

In [4]:
#Task 2: FCFNN
X_train_flat = X_train.reshape(-1, 784)
X_test_flat  = X_test.reshape(-1, 784)

timer_fcfnn = EpochTimer()

fcfnn = Sequential([
    Input(shape=(784,)),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

fcfnn.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
fcfnn.summary()

history_fcfnn = fcfnn.fit(
    X_train_flat, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_flat, y_test),
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True), timer_fcfnn],
    verbose=1
)

print(f"\nAvg time per epoch: {np.mean(timer_fcfnn.times):.2f}s")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 235,146 (918.54 KB)

 Trainable params: 235,146 (918.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.7894 - loss: 0.5847 - val_accuracy: 0.8341 - val_loss: 0.4428
Epoch 2/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8449 - loss: 0.4323 - val_accuracy: 0.8537 - val_loss: 0.4042
Epoch 3/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8572 - loss: 0.3941 - val_accuracy: 0.8653 - val_loss: 0.3833
Epoch 4/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8638 - loss: 0.3706 - val_accuracy: 0.8639 - val_loss: 0.3706
Epoch 5/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8684 - loss: 0.3577 - val_accuracy: 0.8667 - val_loss: 0.3727
Epoch 6/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8743 - loss: 0.3425 - val_accuracy: 0.8758 - val_loss: 0.3470
Epoch 7/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8777 - loss: 0.3321 - val_accuracy: 0.8776 - val_loss: 0.3418
Epoch 8/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8814 - loss: 0.3226 - val_accuracy: 0.

# Task 3
### CNN
- Reshape the input images to 4D tensors with shape `(num_samples, 28, 28, 1)` to include the channel dimension
- Build a CNN using the `Sequential()` API from `keras` using a suitable combination of the following layers
  - `Conv2D()` layers with suitable number of units of suitable size and deafult stride
  - `MaxPooling2D()` layers with suitable size and default stride
  - `Flatten()` layer for feeding feature maps into additional `Dense()` and output layers
  - `Dense()` layers with suitable number of neurons
  - `Dropout()` layers with suitable rates
  - `Dense()` output layer with 10 neurons and `'softmax'` activation
  - Begin with ReLU activation for all layers except the output layer
- Study the model architecture using the `summary()` method
- Compile the model using `Adam()` optimizer, `'categorical_crossentropy'` loss, `'accuracy'` as evaluation metric
- Train the model for suitable number of epochs with a suitable batch size, and provide the validation data separately during training
- Limit training of the model using `EarlyStopping()` from `keras` with a suitable tolerance
- Feel free to change the architecture of the model (number of layers, number of units, activation functions, size and stride of kernels, batch size, number of epochs, early stopping specifics, optimizer learning rate, and so on) and try to improve the model
- Note down the final specifics of the model such as parameter count, average training time per epoch, and performance

In [5]:
#Task 3: CNN
X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_test_cnn  = X_test.reshape(-1, 28, 28, 1)

timer_cnn = EpochTimer()

cnn = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

cnn.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
cnn.summary()

history_cnn = cnn.fit(
    X_train_cnn, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_cnn, y_test),
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True), timer_cnn],
    verbose=1
)

print(f"\nAvg time per epoch: {np.mean(timer_cnn.times):.2f}s")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8071 - loss: 0.5318 - val_accuracy: 0.8667 - val_loss: 0.3677
Epoch 2/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8724 - loss: 0.3505 - val_accuracy: 0.8794 - val_loss: 0.3251
Epoch 3/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8889 - loss: 0.3034 - val_accuracy: 0.8899 - val_loss: 0.2999
Epoch 4/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9014 - loss: 0.2710 - val_accuracy: 0.8970 - val_loss: 0.2760
Epoch 5/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9088 - loss: 0.2496 - val_accuracy: 0.9023 - val_loss: 0.2650
Epoch 6/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9131 - loss: 0.2321 - val_accuracy: 0.9058 - val_loss: 0.2599
Epoch 7/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9215 - loss: 0.2119 - val_accuracy: 0.9076 - val_loss: 0.2517
Epoch 8/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9272 - loss: 0.1975 - val_accuracy: 0

# Task 4
### RCNN
- Use the same input in this model as for CNN
- Build an RCNN using the `Sequential()` API from `keras` using a suitable combination of the following layers
  - `Conv2D()` layers with suitable number of units of suitable size and default stride
  - `MaxPooling2D()` layers with suitable size and default stride
  - `Reshape()` layer to transition from CNN feature map to RNN input
  - `SimpleRNN()` layers with suitable number of units
  - `Dense()` layers with suitable number of neurons
  - `Dropout()` layers with suitable rates
  - `Dense()` output layer with 10 neurons and `softmax` activation
  - Begin with ReLU activation for all layers except the output layer
- Study the model architecture using the `summary()` method
- Compile the model using `Adam()` optimizer, `'categorical_crossentropy'` loss, `'accuracy'` as evaluation metric
- Train the model for suitable number of epochs with a suitable batch size, and provide the validation data separately during training
- Limit training of the model using `EarlyStopping()` from `keras` with a suitable tolerance
- Feel free to change the architecture of the model (number of layers, number of units, size and stride of kernels, batch size, number of epochs, early stopping specifics, optimizer learning rate, and so on) and try to improve the model
- Note down the final specifics of the model such as parameter count, average training time per epoch, and performance

In [6]:
#Task 4: RCNN excited to see this result
timer_rcnn = EpochTimer()

rcnn = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Reshape((5 * 5, 64)),
    SimpleRNN(64, activation='relu'),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

rcnn.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
rcnn.summary()

history_rcnn = rcnn.fit(
    X_train_cnn, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_cnn, y_test),
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True), timer_rcnn],
    verbose=1
)

print(f"\nAvg time per epoch: {np.mean(timer_rcnn.times):.2f}s")

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,882 (124.54 KB)

 Trainable params: 31,882 (124.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.7269 - loss: 0.7438 - val_accuracy: 0.8157 - val_loss: 0.4988
Epoch 2/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.8303 - loss: 0.4724 - val_accuracy: 0.8518 - val_loss: 0.3987
Epoch 3/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.8532 - loss: 0.4107 - val_accuracy: 0.8563 - val_loss: 0.4050
Epoch 4/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8661 - loss: 0.3776 - val_accuracy: 0.8645 - val_loss: 0.3747
Epoch 5/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.8735 - loss: 0.3554 - val_accuracy: 0.8730 - val_loss: 0.3535
Epoch 6/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8794 - loss: 0.3368 - val_accuracy: 0.8754 - val_loss: 0.3416
Epoch 7/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.8848 - loss: 0.3222 - val_accuracy: 0.8786 - val_loss: 0.3345
Epoch 8/20
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.8873 - loss: 0.3134 - val_accuracy: 0

# Task 5
### Model Comparison
- Compare the three final models that were trained in terms of parameter count, average training time, loss and accuracy for training and validation sets
- Create a simple data frame for readability

In [8]:
# ask 5: Model Comparison
results = pd.DataFrame({
    'Model': ['FCFNN', 'CNN', 'RCNN'],
    'Parameters': [
        fcfnn.count_params(),
        cnn.count_params(),
        rcnn.count_params()
    ],
    'Val Accuracy': [
        max(history_fcfnn.history['val_accuracy']),
        max(history_cnn.history['val_accuracy']),
        max(history_rcnn.history['val_accuracy'])
    ],
    'Train Accuracy': [
        max(history_fcfnn.history['accuracy']),
        max(history_cnn.history['accuracy']),
        max(history_rcnn.history['accuracy'])
    ],
    'Avg Epoch Time (s)': [
        round(np.mean(timer_fcfnn.times), 2),
        round(np.mean(timer_cnn.times), 2),
        round(np.mean(timer_rcnn.times), 2)
    ]
})

results['Val Accuracy'] = results['Val Accuracy'].round(4)
results['Train Accuracy'] = results['Train Accuracy'].round(4)
print(results.to_string(index=False))

Model  Parameters  Val Accuracy  Train Accuracy  Avg Epoch Time (s)
FCFNN      235146        0.8792          0.8902                3.48
  CNN      225034        0.9190          0.9499                4.47
 RCNN       31882        0.8969          0.9059                5.87


# CNNs and RNNs Fashion-MNIST Architecture Comparison

Comparing three neural network architectures on the same dataset to understand
the accuracy vs efficiency trade-offs of each design.

**Dataset:** Fashion-MNIST has 70,000 grayscale images across 10 clothing categories
(60k train / 10k test, 28x28 pixels)

**Models built:**
- FCFNN: fully connected baseline, treats each pixel independently
- CNN : extracts spatial features via convoltional filters
- RCNN: CNN feature extraction followed by RNN sequential reading

## Results

| Model | Parameters | Val Accuracy | Avg Epoch Time |
|-------|-----------|--------------|----------------|
| FCFNN | 235,146 | 87.92% | 3.48s |
| CNN | 225,034 | 91.90% | 4.47s |
| RCNN | 31,882 | 89.69% | 5.87s |

## Key Takeaways

**CNN wins on accuracy**  spatial pattern recognition via conv filters extracts
features a dense layer simply cannot see in flatened pixel vectors.

**RCNN wins on efficiency**: 7x fewer parameters than both other models, yet
outperforms FCFNN. The CNN extracts features, the RNN reads them as 25 sequential
spatial patches. More information per parameter.

**FCFNN is the weakest**  treating 784 pixels as independent inputs ignores all
spatial structure. It needs the most parameters to compensate and stil falls short.

The result confirms a core principle: Architecture matters more than parameter count.